# Arabic Boundary-Set: Arabizi Transliteration Verifier

This notebook checks every prompt in `Arabic boundary-set_ARABIZI.json` against `Arabic Boundary-set_MSA.json` and highlights any entry that still contains Arabic script or looks suspicious.

**How to use:**
1. Upload both JSON files when prompted (Cell 2)
2. Run all cells top-to-bottom
3. Review the side-by-side table in Cell 5
4. Flag errors in Cell 6 and export corrections in Cell 7

In [ ]:
# ── Cell 1: Install & imports ────────────────────────────────────────────────
import re, json, unicodedata
from IPython.display import display, HTML
import ipywidgets as widgets

ARABIC_RE = re.compile(r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]')

def has_arabic(text):
    return bool(ARABIC_RE.search(text))

print("Libraries loaded.")


In [ ]:
# ── Cell 2: Upload both JSON files ───────────────────────────────────────────
from google.colab import files

print("Upload  Arabic boundary-set_ARABIZI.json  then  Arabic Boundary-set_MSA.json")
uploaded = files.upload()   # select BOTH files in the dialog

filenames = list(uploaded.keys())
print("Uploaded:", filenames)

arabizi_fn = next((f for f in filenames if 'ARABIZI' in f), None)
msa_fn     = next((f for f in filenames if 'MSA'     in f), None)

if not arabizi_fn or not msa_fn:
    raise ValueError(f"Could not identify files automatically. Got: {filenames}")

arabizi_data = json.loads(uploaded[arabizi_fn])
msa_data     = json.loads(uploaded[msa_fn])

print(f"\nARAbIZI entries : {len(arabizi_data)}")
print(f"MSA entries     : {len(msa_data)}")
assert len(arabizi_data) == len(msa_data), "Length mismatch!"
print("Length check passed ✓")


In [ ]:
# ── Cell 3: Automated quality checks ─────────────────────────────────────────

results = []   # list of dicts, one per entry

for i, (arz, msa) in enumerate(zip(arabizi_data, msa_data)):
    arz_prompt = arz.get('Prompt', '')
    msa_prompt = msa.get('Prompt', '')

    still_arabic  = has_arabic(arz_prompt)
    empty         = len(arz_prompt.strip()) == 0

    # Length ratio: Arabizi is typically 1.2–2.5× longer than the raw Arabic
    # (Arabic packs consonants; Arabizi writes out vowels + digraphs)
    ratio = len(arz_prompt) / max(len(msa_prompt), 1)
    suspicious_short = ratio < 0.4   # too short – probably untranslated
    suspicious_long  = ratio > 4.0   # way too long – probably response, not prompt

    # Category / label
    category   = arz.get('Category', '')
    safe_label = arz.get('Safe/Unsafe', '')

    flag = still_arabic or empty or suspicious_short
    results.append({
        'index'      : i,
        'category'   : category,
        'safe_label' : safe_label,
        'msa'        : msa_prompt,
        'arabizi'    : arz_prompt,
        'still_arabic': still_arabic,
        'empty'      : empty,
        'ratio'      : round(ratio, 2),
        'suspicious' : flag,
    })

# Summary
total          = len(results)
arabic_count   = sum(1 for r in results if r['still_arabic'])
empty_count    = sum(1 for r in results if r['empty'])
short_count    = sum(1 for r in results if r['ratio'] < 0.4 and not r['still_arabic'])
flagged        = sum(1 for r in results if r['suspicious'])

print("=" * 55)
print(f"Total entries          : {total}")
print(f"Still contain Arabic   : {arabic_count}  {'✓' if arabic_count == 0 else '✗ NEEDS FIX'}")
print(f"Empty prompts          : {empty_count}   {'✓' if empty_count == 0 else '✗ NEEDS FIX'}")
print(f"Suspiciously short (<0.4x MSA length): {short_count}")
print(f"Total flagged          : {flagged}")
print("=" * 55)


In [ ]:
# ── Cell 4: Length-ratio distribution ────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    ratios = [r['ratio'] for r in results]
    plt.figure(figsize=(10, 3))
    plt.hist(ratios, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(0.4, color='red',    linestyle='--', label='lower threshold (0.4)')
    plt.axvline(4.0, color='orange', linestyle='--', label='upper threshold (4.0)')
    plt.xlabel('len(Arabizi) / len(MSA)')
    plt.ylabel('Count')
    plt.title('Arabizi / MSA length ratio distribution')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Most ratios should cluster between 0.8 and 2.5")
except ImportError:
    print("matplotlib not available – skipping plot")


In [ ]:
# ── Cell 5: Side-by-side review table ────────────────────────────────────────
# Shows EVERY entry (all 730). Flagged rows are highlighted red.
# Use the filter below to narrow down to only flagged / specific categories.

SHOW_ONLY_FLAGGED = False   # <-- change to True to show only suspicious rows

rows_html = []
for r in results:
    if SHOW_ONLY_FLAGGED and not r['suspicious']:
        continue
    bg = '#ffe0e0' if r['suspicious'] else '#ffffff'
    warnings = []
    if r['still_arabic']:  warnings.append('<b style="color:red">⚠ Arabic script found</b>')
    if r['empty']:         warnings.append('<b style="color:red">⚠ Empty</b>')
    if r['ratio'] < 0.4:   warnings.append(f'<b style="color:orange">⚠ Short ratio ({r["ratio"]})</b>')
    warn_str = ' &nbsp;|&nbsp; '.join(warnings) if warnings else '✓'

    rows_html.append(f"""
    <tr style='background:{bg}'>
      <td style='padding:6px;border:1px solid #ccc;font-weight:bold'>{r['index']}</td>
      <td style='padding:6px;border:1px solid #ccc;font-size:12px;color:#555'>{r['category']}</td>
      <td style='padding:6px;border:1px solid #ccc;font-size:12px'>{r['safe_label']}</td>
      <td style='padding:6px;border:1px solid #ccc;direction:rtl;font-size:13px'>{r['msa'][:200]}</td>
      <td style='padding:6px;border:1px solid #ccc;font-size:12px'>{r['arabizi'][:300]}</td>
      <td style='padding:6px;border:1px solid #ccc;font-size:11px'>{r['ratio']}</td>
      <td style='padding:6px;border:1px solid #ccc;font-size:11px'>{warn_str}</td>
    </tr>""")

table_html = f"""
<div style='overflow-x:auto'>
<table style='border-collapse:collapse;width:100%;font-family:monospace'>
  <thead>
    <tr style='background:#2c3e50;color:white'>
      <th style='padding:8px;border:1px solid #ccc'>Idx</th>
      <th style='padding:8px;border:1px solid #ccc'>Category</th>
      <th style='padding:8px;border:1px solid #ccc'>Safe?</th>
      <th style='padding:8px;border:1px solid #ccc'>MSA (Arabic original)</th>
      <th style='padding:8px;border:1px solid #ccc'>Arabizi transliteration</th>
      <th style='padding:8px;border:1px solid #ccc'>Ratio</th>
      <th style='padding:8px;border:1px solid #ccc'>Status</th>
    </tr>
  </thead>
  <tbody>
    {''.join(rows_html)}
  </tbody>
</table>
</div>
Showing {len(rows_html)} / {total} entries
"""

display(HTML(table_html))


In [ ]:
# ── Cell 6: Manual correction widget ─────────────────────────────────────────
# For any entry you want to fix, enter its index below and type the correct Arabizi.
# Corrections are collected in the `corrections` dict.

corrections = {}

def make_correction_widget(idx):
    entry = results[idx]
    print(f"\n─── Index {idx} ───")
    print(f"Category  : {entry['category']}")
    print(f"MSA       : {entry['msa']}")
    print(f"Current   : {entry['arabizi']}")
    print()
    txt = widgets.Textarea(
        value=entry['arabizi'],
        layout=widgets.Layout(width='90%', height='80px'),
        description=f'idx {idx}:'
    )
    btn = widgets.Button(description='Save correction', button_style='warning')
    out = widgets.Output()

    def on_save(b):
        corrections[idx] = txt.value
        with out:
            print(f"✓ Saved correction for index {idx}")

    btn.on_click(on_save)
    display(widgets.VBox([txt, btn, out]))

# ── Usage: call make_correction_widget(INDEX) for any index you want to fix
# Example:  make_correction_widget(14)
# Example:  make_correction_widget(270)

# Auto-display widgets for all still-flagged entries:
flagged_indices = [r['index'] for r in results if r['suspicious']]
if flagged_indices:
    print(f"Flagged indices: {flagged_indices}")
    for idx in flagged_indices[:10]:   # show first 10 to avoid overwhelming output
        make_correction_widget(idx)
else:
    print("No flagged entries — file looks clean! Use make_correction_widget(N) to edit any entry manually.")


In [ ]:
# ── Cell 7: Export corrected JSON ─────────────────────────────────────────────
if corrections:
    print(f"Applying {len(corrections)} manual corrections...")
    for idx, new_text in corrections.items():
        arabizi_data[idx]['Prompt'] = new_text
        print(f"  ✓ index {idx}")

# Final Arabic-script check
remaining = [i for i, item in enumerate(arabizi_data) if has_arabic(item.get('Prompt',''))]
print(f"\nEntries still containing Arabic script after corrections: {len(remaining)}")
if remaining:
    print("Remaining indices:", remaining)

# Save
import io
out_json = json.dumps(arabizi_data, ensure_ascii=False, indent=2)
with open('Arabic_boundary-set_ARABIZI_verified.json', 'w', encoding='utf-8') as f:
    f.write(out_json)

files.download('Arabic_boundary-set_ARABIZI_verified.json')
print("\nDownloaded: Arabic_boundary-set_ARABIZI_verified.json")
